In [14]:
import pandas as pd 
import polars as pl
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# 1
import requests
from io import BytesIO

import polars_talib as plta

# 1:    Data Organisation

In [2]:
def get_nifty500():
    """
    Download current NIFTY 500 constituents from NSE
    and return a normalized DataFrame.
    """

    url = "https://nsearchives.nseindia.com/content/indices/ind_nifty500list.csv"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/151.0.0.0 Safari/537.36"
        ),
        "Accept": "text/csv,application/csv,*/*",
        "Referer": "https://www.nseindia.com/"
    }

    session = requests.Session()
    session.headers.update(headers)

    response = session.get(url, timeout=30)
    response.raise_for_status()

    df = pd.read_csv(BytesIO(response.content))

    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("&", "and")
    )

    # Normalize string columns
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.strip()

    # Normalize symbol
    if "symbol" in df.columns:
        df["symbol"] = df["symbol"].str.upper()

    # Remove duplicate symbols
    df = df.drop_duplicates(subset="symbol")

    # Sort alphabetically
    df = df.sort_values("symbol").reset_index(drop=True)

    return df

In [3]:
# Fetching Nifty500 constisuents
nifty500 = get_nifty500()

print(nifty500.shape)
print(nifty500.head())

(501, 5)
                  company_name            industry     symbol series  \
0             360 ONE WAM Ltd.  Financial Services     360ONE     EQ   
1                3M India Ltd.         Diversified    3MINDIA     EQ   
2  Aadhar Housing Finance Ltd.  Financial Services  AADHARHFC     EQ   
3        Aarti Industries Ltd.           Chemicals   AARTIIND     EQ   
4        Aavas Financiers Ltd.  Financial Services      AAVAS     EQ   

      isin_code  
0  INE466L01038  
1  INE470A01017  
2  INE883F01010  
3  INE769A01020  
4  INE216P01012  


In [4]:
universe= nifty500["symbol"].to_list()
universe[:5]

['360ONE', '3MINDIA', 'AADHARHFC', 'AARTIIND', 'AAVAS']

In [5]:
# Fetching price data from yfinance

tickers= [f"{tic}.NS" for tic in universe]
df = yf.download(tickers, auto_adjust= True, period= "1y", group_by= "column", threads= True)
df= df.stack(level= "Ticker", future_stack= True).reset_index()
df= df.drop(columns= ["Adj Close"])
df= df.rename(columns= {"Ticker": "symbol"})
df["symbol"]= df["symbol"].str.replace(".NS", "", regex= False)         #.str.upper().str.strip() ADDITIONAL CHECK
df["Date"]= pd.to_datetime(df["Date"])                                  # Normalise date columns
df.columns= [col.lower().replace(" ", "_") for col in df.columns]        # Clean standardised column names lower case and _
df= df.sort_values(["symbol", "date"]).reset_index(drop= True)

# Bridge pandas dataframe to polars
df= pl.from_pandas(df)

[*********************100%***********************]  501 of 501 completed

1 Failed download:
['DUMMYHEG.NS']: HTTPError('HTTP Error 404: ')


In [6]:
df

date,symbol,close,high,low,open,volume
datetime[ms],str,f64,f64,f64,f64,f64
2025-09-09 00:00:00,"""360ONE""",1032.623291,1038.657372,1026.787002,1028.864292,819360.0
2025-09-10 00:00:00,"""360ONE""",1066.354736,1070.707229,1032.326324,1032.623131,1.184367e6
2025-09-11 00:00:00,"""360ONE""",1045.680664,1063.387324,1043.108775,1061.408929,430699.0
2025-09-12 00:00:00,"""360ONE""",1057.254272,1059.430459,1036.876709,1054.484471,343905.0
2025-09-15 00:00:00,"""360ONE""",1067.343994,1071.201888,1047.560046,1047.658942,353507.0
…,…,…,…,…,…,…
2026-09-03 00:00:00,"""ZYDUSWELL""",538.200012,541.400024,531.099976,538.0,135712.0
2026-09-04 00:00:00,"""ZYDUSWELL""",531.700012,546.599976,527.150024,536.950012,209449.0
2026-09-07 00:00:00,"""ZYDUSWELL""",555.349976,581.0,536.650024,537.299988,7.627303e6


We are going to label out target variables:

Y= 1;   if Max Return > 4% && Max Draw Down > -2%
y= 0;   else y = 0

In [7]:
# Future Horizon Extraction
horizon= 9
tp= 0.1
sl= 0.07

future_exprs= []
for day in range(1, horizon+1):
    future_exprs.extend([
        pl.col("open").shift(-day).over("symbol").alias(f"future_open_{day}"),
        pl.col("high").shift(-day).over("symbol").alias(f"future_high_{day}"),
        pl.col("low").shift(-day).over("symbol").alias(f"future_low_{day}"),
        pl.col("close").shift(-day).over("symbol").alias(f"future_close_{day}"),
    ])

df= df.with_columns(future_exprs)

In [8]:
# Base Metrics & Max Returns
future_close_cols= [f"future_close_{day}" for day in range(1, horizon +1)]
high_cols= [f"future_high_{day}" for day in range(1, horizon +1)]

df= df.with_columns(
    days_available= pl.sum_horizontal(
        [pl.col(c).is_not_null() for c in future_close_cols]
    ),
    entry= pl.col("future_open_1"),
).with_columns(
    tp_price= pl.col("entry") *(1 +tp),
    max_returns_9d=(
        pl.max_horizontal([pl.col(c) for c in high_cols]) /pl.col("entry")
    )
    -1,
)

In [9]:
# Path Drawdown & Event Simulation
entry_arr= df["entry"].to_numpy()
tp_price_arr= df["tp_price"].to_numpy()
n_rows= len(df)

event= np.full(n_rows, "NONE", dtype= object)
event_day= np.full(n_rows, np.nan)
running_peak= entry_arr.copy()
unresolved=  np.ones(n_rows, dtype=bool)

dd_matrix= np.zeros((n_rows, horizon))

for idx, day in enumerate(range(1, horizon+1)):
    high_arr= df[f"future_high_{day}"].to_numpy()
    low_arr= df[f"future_low_{day}"].to_numpy()

    running_peak= np.maximum(running_peak, high_arr)
    dd= low_arr /running_peak -1
    dd_matrix[:, idx]= dd

    tp_hit= high_arr >= tp_price_arr
    sl_hit= low_arr <= running_peak *(1 -sl)
    active= unresolved

    both= active & tp_hit & sl_hit
    tp_only= active & tp_hit & ~sl_hit
    sl_only= active & sl_hit & ~tp_hit

    event[both]= "BOTH"
    event_day[both]= day
    event[tp_only]= "TP"
    event_day[tp_only]= day
    event[sl_only]= "SL"
    event_day[sl_only]= day

    resolved= both | tp_only | sl_only
    unresolved[resolved]= False

# Assign results back to polars
df= df.with_columns(
    max_dd_9d= dd_matrix.min(axis= 1),
    first_event= event,
    event_day= event_day,
).with_columns(
    censored= (pl.col("first_event").eq("NONE"))
    & (pl.col("days_available") <horizon)
)

# Finally the targets
df= df.with_columns(
    target= pl.when(pl.col("first_event").eq("TP"))
    .then(1)
    .when(pl.col("first_event").eq("SL"))
    .then(-1)
    .when(pl.col("censored"))
    .then(None)
    .otherwise(0)
)

In [10]:
# Concurency & Sampling weights

df= df.with_columns(
    span_days= pl.coalesce([pl.col("event_day"), pl.lit(horizon)])
    .fill_null(horizon)
    .fill_nan(horizon)
    .cast(pl.Int64)
).with_columns(
    start_epoch= pl.col("date").dt.epoch(time_unit= "d"),
    end_epoch= pl.col("date").dt.epoch(time_unit= "d") +pl.col("span_days"),
)

starts= df["start_epoch"].to_numpy()
ends= df["end_epoch"].to_numpy()

unique_days= np.unique(np.concatenate([starts, ends]))
concurrency_map= {}
for day in unique_days:
    active_count= np.sum((starts <= day) & (ends >= day))
    concurrency_map[day]= active_count

weights= []
for s, e in zip(starts, ends):
    trade_days= np.arange(s, e+1)
    uniqueness_sum= sum(
        1.0 /concurrency_map.get(d, 1) for d in trade_days if d in concurrency_map
    )
    weights.append(uniqueness_sum /len(trade_days) if len(trade_days) > 0 else 1.0)

df= df.with_columns(sample_weight= pl.Series(weights))

In [11]:
df.select(pl.col("target").value_counts(sort=True)).unnest("target")

target,count
i32,u32
0,61724
-1,50755
1,8036
null,5737


In [12]:
df

date,symbol,close,high,low,open,volume,future_open_1,future_high_1,future_low_1,future_close_1,future_open_2,future_high_2,future_low_2,future_close_2,future_open_3,future_high_3,future_low_3,future_close_3,future_open_4,future_high_4,future_low_4,future_close_4,future_open_5,future_high_5,future_low_5,future_close_5,future_open_6,future_high_6,future_low_6,future_close_6,future_open_7,future_high_7,future_low_7,future_close_7,future_open_8,future_high_8,future_low_8,future_close_8,future_open_9,future_high_9,future_low_9,future_close_9,days_available,entry,tp_price,max_returns_9d,max_dd_9d,first_event,event_day,censored,target,span_days,start_epoch,end_epoch,sample_weight
datetime[ms],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,str,f64,bool,i32,i64,i32,i64,f64
2025-09-09 00:00:00,"""360ONE""",1032.623291,1038.657372,1026.787002,1028.864292,819360.0,1032.623131,1070.707229,1032.326324,1066.354736,1061.408929,1063.387324,1043.108775,1045.680664,1054.484471,1059.430459,1036.876709,1057.254272,1047.658942,1071.201888,1047.560046,1067.343994,1067.343968,1085.248417,1066.651578,1079.313232,1078.917541,1095.635001,1077.730552,1091.084717,1091.08476,1093.063155,1082.379774,1090.194458,1090.392379,1092.073966,1063.38724,1069.124634,1059.430421,1075.257579,1030.941487,1039.052979,9,1032.623131,1135.885444,0.061021,-0.059047,"""NONE""",NaN,false,0,9,20340,20349,0.000654
2025-09-10 00:00:00,"""360ONE""",1066.354736,1070.707229,1032.326324,1032.623131,1.184367e6,1061.408929,1063.387324,1043.108775,1045.680664,1054.484471,1059.430459,1036.876709,1057.254272,1047.658942,1071.201888,1047.560046,1067.343994,1067.343968,1085.248417,1066.651578,1079.313232,1078.917541,1095.635001,1077.730552,1091.084717,1091.08476,1093.063155,1082.379774,1090.194458,1090.392379,1092.073966,1063.38724,1069.124634,1059.430421,1075.257579,1030.941487,1039.052979,1036.678925,1041.92172,1004.431112,1008.585693,9,1061.408929,1167.549821,0.032246,-0.083243,"""SL""",9.0,false,-1,9,20341,20350,0.000482
2025-09-11 00:00:00,"""360ONE""",1045.680664,1063.387324,1043.108775,1061.408929,430699.0,1054.484471,1059.430459,1036.876709,1057.254272,1047.658942,1071.201888,1047.560046,1067.343994,1067.343968,1085.248417,1066.651578,1079.313232,1078.917541,1095.635001,1077.730552,1091.084717,1091.08476,1093.063155,1082.379774,1090.194458,1090.392379,1092.073966,1063.38724,1069.124634,1059.430421,1075.257579,1030.941487,1039.052979,1036.678925,1041.92172,1004.431112,1008.585693,1010.959805,1021.840977,994.242344,1015.411194,9,1054.484471,1159.932918,0.039024,-0.092542,"""SL""",8.0,false,-1,8,20342,20350,0.000424
2025-09-12 00:00:00,"""360ONE""",1057.254272,1059.430459,1036.876709,1054.484471,343905.0,1047.658942,1071.201888,1047.560046,1067.343994,1067.343968,1085.248417,1066.651578,1079.313232,1078.917541,1095.635001,1077.730552,1091.084717,1091.08476,1093.063155,1082.379774,1090.194458,1090.392379,1092.073966,1063.38724,1069.124634,1059.430421,1075.257579,1030.941487,1039.052979,1036.678925,1041.92172,1004.431112,1008.585693,1010.959805,1021.840977,994.242344,1015.411194,1013.927413,1018.081994,1001.067845,1011.058716,9,1047.658942,1152.424836,0.045794,-0.092542,"""SL""",7.0,false,-1,7,20343,20350,0.000394
2025-09-15 00:00:00,"""360ONE""",1067.343994,1071.201888,1047.560046,1047.658942,353507.0,1067.343968,1085.248417,1066.651578,1079.313232,1078.917541,1095.635001,1077.730552,1091.084717,1091.08476,1093.063155,1082.379774,1090.194458,1090.392379,1092.073966,1063.38724,1069.124634,1059.430421,1075.257579,1030.941487,1039.052979,1036.678925,1041.92172,1004.431112,1008.585693,1010.959805,1021.840977,994.242344,1015.411194,1013.927413,1018.081994,1001.067845,1011.058716,1005.024582,1013.135952,986.031979,990.977966,9,1067.343968,1174.078365,0.026506,-0.100036,"""SL""",6.0,false,-1,6,20346,20352,0.000329
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [16]:
# 1. Ensure strict temporal sorting before calculating rolling windows
df = df.sort(["symbol", "date"])

# 2. Feature Engineering Pipeline (Volatility, Momentum, Volume, Structure)
df = df.with_columns(
    [
        # Volatility: Normalized Average True Range (NATR)
        plta.natr(
            pl.col("high"), pl.col("low"), pl.col("close"), timeperiod=14
        )
        .over("symbol")
        .alias("natr_14"),
        # Momentum: Rate of Change (ROC)
        plta.roc(pl.col("close"), timeperiod=10)
        .over("symbol")
        .alias("roc_10d"),
        # Volume: Relative Volume (RVOL)
        (pl.col("volume") / pl.col("volume").rolling_mean(20))
        .over("symbol")
        .alias("rvol_20d"),
        # Structure: Donchian Channel Location
        (
            (pl.col("close") - pl.col("low").rolling_min(20))
            / (
                pl.col("high").rolling_max(20)
                - pl.col("low").rolling_min(20)
                + 1e-8
            )
        )
        .over("symbol")
        .alias("donchian_loc_20d"),
    ]
)

# 3. Cross-Sectional Ranking (Regime-agnostic daily relative sorting)
df = df.with_columns(
    [
        pl.col("roc_10d").rank(descending=True).over("date").alias("rank_roc_10d"),
        pl.col("natr_14")
        .rank(descending=False)
        .over("date")
        .alias("rank_natr_14"),  # Lower vol ranked higher
        pl.col("rvol_20d")
        .rank(descending=True)
        .over("date")
        .alias("rank_rvol_20d"),
    ]
)

# 4. The Leakage Shift (Preventing look-ahead bias by lagging features by 1 row)
feature_cols = [
    "rank_roc_10d",
    "rank_natr_14",
    "rank_rvol_20d",
    "donchian_loc_20d",
]

df = df.with_columns(
    [pl.col(col).shift(1).over("symbol") for col in feature_cols]
)

ComputeError: the plugin failed with message: Could not compute indicator, err: TA_OUT_OF_RANGE_END_INDEX